# 02 --  Data Preprocessing

## Concept
Raw data cannot be fed directly into ML models. Preprocessing transforms raw features into clean numerical representations.

## Mathematical Intuition
- **StandardScaler**: z = (x - mean) / std --  zero mean, unit variance
- **MinMaxScaler**: x' = (x - min) / (max - min) --  range [0, 1]
- **One-Hot Encoding**: each category -> binary column
- **Missing value imputation**: mean/median/mode replacement

## Interview Questions
1. When would you use StandardScaler vs MinMaxScaler?
2. Why do tree-based models not require feature scaling?
3. What is the dummy variable trap and how to avoid it?

## Production Mapping
Production preprocessing lives in `pipeline/preprocessing.py` as a sklearn ColumnTransformer.


In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
np.random.seed(42)
n = 200
df = pd.DataFrame({
    'capacity_mw': np.random.exponential(500, n),
    'region': np.random.choice(['NA', 'EU', 'APAC', 'ME'], n),
    'is_active': np.random.choice([True, False], n),
    'age_years': np.random.exponential(30, n),
})
df.loc[::10, 'capacity_mw'] = np.nan  # inject missing
df.head()

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, ['capacity_mw', 'age_years']),
    ('cat', cat_pipeline, ['region']),
])
X = preprocessor.fit_transform(df)
print(f"Shape after preprocessing: {X.shape}")
print(f"Feature names: {preprocessor.get_feature_names_out()}")

In [ ]:
# Compare scalers
raw = df['age_years'].dropna().values.reshape(-1, 1)
standardized = StandardScaler().fit_transform(raw)
minmaxed = MinMaxScaler().fit_transform(raw)
print(f"Raw: mean={raw.mean():.2f}, std={raw.std():.2f}")
print(f"Standard: mean={standardized.mean():.2f}, std={standardized.std():.2f}")
print(f"MinMax: min={minmaxed.min():.2f}, max={minmaxed.max():.2f}")

## Key Takeaways
- Numerical features: impute -> scale
- Categorical features: impute -> encode
- Boolean features: impute (fillna) -> keep as 0/1
- Timestamp features: extract cyclical components (sin/cos of hour, day_of_week, month)
- Build preprocessing as a Pipeline for reproducibility